# V2.5-P on a separate GPU
Upload CCG-LM_source.zip and precision_study_cache.zip. This notebook contains no Blackwell results. Every GPU must use a separate output directory.

In [ ]:
from google.colab import files
import zipfile
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall('.')

from pathlib import Path
if Path('precision_study_cache.zip').exists():
    with zipfile.ZipFile('precision_study_cache.zip') as z: z.extractall('.')


In [ ]:
import torch,re
print(torch.__version__,torch.version.cuda)
assert torch.cuda.is_available()
assert hasattr(torch.backends.cuda.matmul,'fp32_precision'), 'Matching CUDA PyTorch >=2.9 is required'
print(torch.cuda.get_device_name())
output='reports/cky_v25_precision_'+re.sub('[^a-zA-Z0-9]+','_',torch.cuda.get_device_name()).strip('_')
print(output)


In [ ]:
%pip install -q datasets tokenizers numpy PyYAML pandas matplotlib tqdm pytest


Verify the uploaded cache before comparing hardware. Keep configs and token batches identical. Save checkpoints to persistent storage before Colab disconnects; --resume restores model/optimizer/sampler/RNG/policy/scaler.

In [ ]:
!python -m data.prepare --config configs/precision/stability_smoke.yaml --max-train-stories 32 --max-validation-stories 8 --verify-cache


In [ ]:
!python run_precision_study.py --output-dir {output} --resume


In [ ]:
import pandas as pd
reference=pd.read_csv('reports/cky_v25_precision/benchmark.csv').groupby('case').weight_sha256.first()
measured=pd.read_csv(Path(output)/'benchmark.csv').groupby('case').weight_sha256.first()
assert reference.equals(measured), 'Different weights/runtime initialization: do not attribute a cross-GPU difference to hardware alone'
